In [72]:
import pandas as pd
import psycopg2
import re
from psycopg2.extras import execute_values

In [73]:
conn = psycopg2.connect(
    host="localhost",
    database="TWM",
    port="5432",
    user="postgres",
    password="1234"
)

In [74]:
cursor = conn.cursor()

In [75]:
# ---------- LOAD EXCEL FILE ----------
file_path = "PO10976964_2303_2026_01_05_0802.csv"

df = pd.read_csv(file_path)

print("Columns detected:", df.columns.tolist())

Columns detected: ['origin_trading_partner_name', 'origin_trading_partner_number', 'origin_location_name', 'origin_location_state', 'destination_trading_partner_name', 'destination_location_name', 'destination_location_number', 'destination_location_address1', 'destination_location_address2', 'destination_location_city', 'destination_location_state', 'destination_location_zip', 'destination_location_phone1', 'order_number', 'order_type_name', 'e_delivery_date', 'create_date', 'approve_date', 'twm_item_code', 'vendor_item_code', 'item_name', 'package_type', 'primary_upc', 'defined_vintage', 'order_qty_cs', 'order_qty_units', 'total_order_qty_units', 'order_units_case', 'order_cost', 'deposit', 'deposit_extension', 'bottle_charge', 'bottle_charge_extension', 'cost_extension']


In [76]:
# ---------- KEEP ONLY REQUIRED COLUMNS ----------
required_columns = [
    "vendor_item_code",
    "twm_item_code",
    "item_name",
    "package_type",
    "order_cost"
]
df = df[required_columns]

In [77]:
package_map = {
    "1.75":6,
    "1.75L":6,
    "1.75l":6,
    "1.75l":6,
    "100ml":48,
    "10nr-4p": 6,
    "11nr-18L":1,
    "11nr-18l":1,
    "11nr-24L":1,
    "11nr-24l":1,
    "11nr-6p": 2,
    "12c-12p":2,
    "12c-15p":2,
    "12c-18L": 1,
    "12c-18p":1,
    "12c-24L":1,
    "12c-24p":1,
    "12c-30p": 1,
    "12c-4p":6,
    "12c-6p": 4,
    "12c-8p": 3,
    "12nr-12p":2,
    "12nr-24l":1,
    "12c-24l":1,
    "12nr-18L":1,
    "12nr-18l":1,
    "12c-18l":1,
    "12nr-24L":1,
    "12nr-6p": 4,
    "16al-12p":2,
    "16al-8p": 3,
    "16c-18L": 1,
    "16c-18l":1,
    "16c-6p":4,
    "16c-8p":3,
    "16c-4p":6,
    "19c-15L":15,
    "19c-15l":15,
    "25c-15l":15,
    "25c-15L":15,
    "1L":12,
    "1l":12,
    "200ml":24,
    "375ml":24,
    "50-10gft":12,
    "100-6gft":20,
    "50-20gft":6,
    "500ml":12,
    "500ml-6p":2,
    "50ml":120,
    "50ml-4gf":30,
    "50ml-6gf":20,
    "50ml10gf":12,
    "50ml15gf":12,
    "7.5c-12p":2,
    "700ml":6,
    "720ml":12,
    "750ml":12,
    "8c-4p":6,
    "16oz":12
}

df["sellable_packs"] = df["package_type"].map(package_map)

df["sellable_packs"] = df["sellable_packs"].where(pd.notnull(df["sellable_packs"]), None)

In [78]:
records = df[[
    "vendor_item_code",
    "twm_item_code",
    "item_name",
    "package_type",
    "order_cost",
    "sellable_packs"
]].values.tolist()

In [79]:

for i, row in enumerate(records):
    try:
        cursor.execute("""
            INSERT INTO po_import
            (vendor_item_code, twm_item_code, item_name, package_type, order_cost, sellable_packs)
            VALUES (%s, %s, %s, %s, %s, %s)
        """, row)
#Debug line. Run script if errors occur during import, breaks script and shows data on what fails to import.
    except Exception as e:
        print(f"\n Error on row {i}: {row}")
        print(e)
        break

In [80]:
conn.commit()
cursor.close()
conn.close()

In [81]:
print("Import complete.")

Import complete.


In [82]:
#Run within SQL after importing new CSVs.

#INSERT INTO products
#(vendor_item_code, item_name, package_type, order_cost, sellable_packs)
#SELECT DISTINCT ON (vendor_item_code, LOWER(TRIM(package_type)))
#    TRIM(vendor_item_code),
#    twm_item_code,
#    item_name,
#    LOWER(TRIM(package_type)),
#    order_cost,
#    sellable_packs
#FROM po_import
#ORDER BY 
#    vendor_item_code,
#    
#    LOWER(TRIM(package_type)),
#    order_cost DESC
#ON CONFLICT (vendor_item_code, package_type)
#DO UPDATE SET
#    order_cost = EXCLUDED.order_cost,
#    sellable_packs = EXCLUDED.sellable_packs;